# Presentation 02 — Model results

Figures describing **how well the trained model works on held-out validation data**: training curves (if saved), overall accuracy, confusion matrix, per-class precision/recall/F1.

Run cells in order. All generated figures are saved to `outputs/presentation/` on your Drive so you can drop them straight into slides.

## Setup — mount Drive, clone latest code, configure paths

This block is the same across all presentation notebooks. It mounts your Drive, pulls the latest branch, adds `src/` to `sys.path` and sets `PRIMATE_DATA_ROOT`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf primates-sound-detection
!git clone -q -b claude/review-repo-QAc6j https://github.com/Mo119m/primates-sound-detection.git
%cd /content/primates-sound-detection
!pip install -q -r requirements.txt

In [ ]:
import os, sys, importlib
PROJECT_ROOT = '/content/primates-sound-detection'
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
os.environ['PRIMATE_DATA_ROOT'] = '/content/drive/MyDrive/primates-data'

import config
importlib.reload(config)

PRESENT_DIR = os.path.join(config.OUTPUT_ROOT, 'presentation')
os.makedirs(PRESENT_DIR, exist_ok=True)
print('Presentation figures will be saved to:', PRESENT_DIR)

In [ ]:
# Consistent matplotlib style for every figure in this notebook
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 220,
    'savefig.bbox': 'tight',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'legend.frameon': False,
})

# Shared species color palette
SPECIES_COLORS = {
    'Cercopithecus_nictitans': '#E67E22',  # orange
    'Colobus_guereza':         '#27AE60',  # green
    'Pan_troglodytes':         '#2980B9',  # blue
    'Background':              '#7F8C8D',  # grey
}

def save_fig(fig, name):
    path = os.path.join(PRESENT_DIR, name)
    fig.savefig(path)
    print(f'  saved -> {path}')
    return path

## Step 1 — Load the trained model

In [ ]:
import model as model_module
importlib.reload(model_module)

best_model_path = os.path.join(config.MODEL_SAVE_DIR, 'best_model.h5')
assert os.path.exists(best_model_path), f'Trained model not found at {best_model_path} — run the main pipeline first.'

model = model_module.load_trained_model(best_model_path)
print('Model loaded.')

## Figure 1 — Training curves (if training history is saved)

`train.run_complete_training_pipeline()` writes `training_history.json` next to `best_model.h5`. If that file is there, this cell plots accuracy/loss vs epoch. If it isn't, this cell just prints a message and skips.

In [ ]:
import json
import numpy as np

history_path = os.path.join(config.MODEL_SAVE_DIR, 'training_history.json')

if not os.path.exists(history_path):
    print(f'No training_history.json at {history_path} — skip training curves figure.')
    print('(To get this figure you must run the main notebook with FORCE_RETRAIN=True once.)')
else:
    with open(history_path) as f:
        history = json.load(f)
    epochs = np.arange(1, len(history['loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].plot(epochs, history['accuracy'], marker='o', label='train')
    axes[0].plot(epochs, history['val_accuracy'], marker='s', label='val')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Training / validation accuracy')
    axes[0].set_ylim(0, 1.02)
    axes[0].legend()

    axes[1].plot(epochs, history['loss'], marker='o', label='train')
    axes[1].plot(epochs, history['val_loss'], marker='s', label='val')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Training / validation loss')
    axes[1].legend()

    plt.tight_layout()
    save_fig(fig, '01_training_curves.png')
    plt.show()

## Step 2 — Rebuild the validation split and run predictions

We reproduce the exact same train/val split by calling `train.prepare_dataset()` with the fixed `RANDOM_SEED`. **This is slow** (loads + augments all audio) but gives us reproducible metrics.

If this OOMs on a small GPU, drop `BATCH_SIZE` in `src/config.py` first.

In [ ]:
import train
importlib.reload(train)

X_train, X_val, y_train, y_val, class_names = train.prepare_dataset()
print(f'\nValidation set: {len(X_val)} samples')

In [ ]:
y_prob = model.predict(X_val, batch_size=config.BATCH_SIZE, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

overall_acc = float((y_pred == y_val).mean())
top2 = np.argsort(y_prob, axis=1)[:, -2:]
top2_acc = float(np.mean([y_val[i] in top2[i] for i in range(len(y_val))]))
print(f'Validation accuracy       : {overall_acc:.4f}')
print(f'Validation top-2 accuracy : {top2_acc:.4f}')

## Figure 2 — Confusion matrix (absolute + row-normalized)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_val, y_pred, labels=list(range(len(class_names))))
cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, mat, title, fmt in [
    (axes[0], cm,      'Counts',       'd'),
    (axes[1], cm_norm, 'Row-normalized (recall)', '.2f'),
]:
    im = ax.imshow(mat, cmap='Blues', aspect='auto', vmin=0,
                   vmax=mat.max() if mat.max() > 0 else 1)
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=30, ha='right')
    ax.set_yticklabels(class_names)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    ax.grid(False)
    thresh = mat.max() / 2 if mat.max() > 0 else 0.5
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            ax.text(j, i, format(val, fmt),
                    ha='center', va='center',
                    color='white' if val > thresh else 'black',
                    fontsize=11, fontweight='bold')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(f'Confusion matrix — val accuracy {overall_acc:.1%}', fontsize=15, y=1.02)
plt.tight_layout()
save_fig(fig, '02_confusion_matrix.png')
plt.show()

## Figure 3 — Per-class precision / recall / F1

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, classification_report

p, r, f1, support = precision_recall_fscore_support(
    y_val, y_pred, labels=list(range(len(class_names))), zero_division=0
)

import pandas as pd
report_df = pd.DataFrame({
    'class': class_names,
    'precision': p,
    'recall': r,
    'f1': f1,
    'support': support,
})
print(report_df.to_string(index=False))
report_df.to_csv(os.path.join(PRESENT_DIR, '03_classification_report.csv'), index=False)

x = np.arange(len(class_names))
w = 0.27
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.bar(x - w, p,  w, label='Precision', color='#3498DB')
ax.bar(x,     r,  w, label='Recall',    color='#E67E22')
ax.bar(x + w, f1, w, label='F1',        color='#2ECC71')
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=20, ha='right')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.set_title('Per-class precision / recall / F1 on validation set')
ax.legend(loc='lower right')
for i, (pi, ri, fi) in enumerate(zip(p, r, f1)):
    for xi, val in [(i - w, pi), (i, ri), (i + w, fi)]:
        ax.text(xi, val + 0.01, f'{val:.2f}', ha='center', fontsize=9)
plt.tight_layout()
save_fig(fig, '03_per_class_prf.png')
plt.show()

## Figure 4 — Overall accuracy headline (big-number card)

Stand-alone 'one big number' figure you can drop on a summary slide.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.axis('off')
ax.text(0.5, 0.72, f'{overall_acc:.1%}', ha='center', va='center',
        fontsize=80, fontweight='bold', color='#2C3E50')
ax.text(0.5, 0.34, 'Validation accuracy', ha='center', va='center',
        fontsize=20, color='#7F8C8D')
ax.text(0.5, 0.12, f'Top-2 accuracy: {top2_acc:.1%}   |   {len(y_val)} validation samples',
        ha='center', va='center', fontsize=14, color='#95A5A6')
save_fig(fig, '04_headline_accuracy.png')
plt.show()

## Done

Figures `01_*.png` through `04_*.png` + the classification report CSV are on Drive under `outputs/presentation/`.